# A little extra!

## New addition to Week 1

### The Unreasonable Effectiveness of the Agent Loop

# What is an Agent?

## Three competing definitions

1. AI systems that can do work for you independently - Sam Altman

2. A system in which an LLM controls the workflow - Anthropic

3. An LLM agent runs tools in a loop to achieve a goal

## The third one is the new, emerging definition

But what does it mean?

Let's make it real.

In [1]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [2]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [3]:
openai = OpenAI()

In [4]:
# Some lists!

todos = []
completed = []

In [5]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]: #if equivalent index found True in completed
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n" # if completed add green strike
        else:
            result += f"Todo #{index + 1}: {todo}\n" # if not completed just show
    show(result)
    return result

In [6]:
get_todo_report()

''

In [7]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions) # Add the todo descripts to the list
    completed.extend([False] * len(descriptions))  # add the equivalent number of Falses to match todos list
    return get_todo_report()

In [8]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(todos): # consider this an and statement
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [9]:
todos, completed = [], []

create_todos(["Go to gym", "Drink Coffee", "Get to work"])
#print(completed)

Todo #1: Go to gym
Todo #2: Drink Coffee
Todo #3: Get to work

'Todo #1: Go to gym\nTodo #2: Drink Coffee\nTodo #3: Get to work\n'

In [10]:
mark_complete(2, "none left")

none left

Todo #1: Go to gym
Todo #2: Drink Coffee
Todo #3: Get to work

'Todo #1: Go to gym\nTodo #2: [green][strike]Drink Coffee[/strike][/green]\nTodo #3: Get to work\n'

In [11]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [12]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [13]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [14]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [15]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [16]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
An experienced resistance weight trainer over 50 years old needs a 2-day full body training plan. 
The individual's primary goal is muscle maintenance, secondary goal is muscle hypertrophy.
This routine should include the most effective science-backed free weight and resistance machine exercises, and each day
should last between 60 and 90 minutes. Include the number of sets and reps, as well as the amount of weight required. 
Design the 2-day plan and label as Workout A and Workout B.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [17]:
todos, completed = [], []
loop(messages)

Todo #1: Set assumptions for an experienced 50+ lifter (weekly frequency, intensity targets, RPE/RIR, warm-up, rest
times) and define how to prescribe weights without exact 1RMs.
Todo #2: Design Workout A (full body) using evidence-backed compound lifts + key accessories; fit into 60–90 min 
with sets/reps/rest.
Todo #3: Design Workout B (full body) with different primary patterns/exercises to balance fatigue and coverage; 
fit into 60–90 min.
Todo #4: Provide weight prescriptions using %1RM / RPE plus example loads (based on reasonable estimated 1RMs) and 
clear progression guidance.
Todo #5: Format final plan in Rich console markup with sets, reps, weight, rest, and brief notes for 50+ 
considerations (joint-friendly, volume).

Assumptions set: 2 non-consecutive days/week (e.g., Mon/Thu). Goal = maintenance with slight hypertrophy: ~8–12 
hard sets/muscle/week, mostly 5–10 rep compounds + 8–15 rep accessories. Intensity: work sets at RPE 7–9 (leave 
~1–3 reps in reserve, RIR). Rest: 2–3 min for big compounds, 60–90s for accessories. Warm-up: 8–12 min (bike/row + 
dynamic mobility) + 2–4 ramp sets for first two lifts. Weight prescription method: use %1RM ranges tied to rep 
targets (e.g., 5–6 reps ≈ 80–85%1RM; 8 reps ≈ 75–80%; 10–12 reps ≈ 65–75%) or choose a load that matches target 
RPE; adjust each set to stay in the rep range at the prescribed RPE.

Todo #1: Set assumptions for an experienced 50+ lifter (weekly frequency, intensity targets, RPE/RIR, warm-up, rest
times) and define how to prescribe weights without exact 1RMs.
Todo #2: Design Workout A (full body) using evidence-backed compound lifts + key accessories; fit into 60–90 min 
with sets/reps/rest.
Todo #3: Design Workout B (full body) with different primary patterns/exercises to balance fatigue and coverage; 
fit into 60–90 min.
Todo #4: Provide weight prescriptions using %1RM / RPE plus example loads (based on reasonable estimated 1RMs) and 
clear progression guidance.
Todo #5: Format final plan in Rich console markup with sets, reps, weight, rest, and brief notes for 50+ 
considerations (joint-friendly, volume).

Workout A built around: squat pattern (joint-tolerant option), horizontal press, vertical pull, hip hinge 
accessory, shoulder/scap stability, arms + calves. Volume set to ~18–22 work sets total to land in 60–90 min with 
appropriate rests; compound-first ordering and optional paired accessories to save time.

Todo #1: Set assumptions for an experienced 50+ lifter (weekly frequency, intensity targets, RPE/RIR, warm-up, rest
times) and define how to prescribe weights without exact 1RMs.
Todo #2: Design Workout A (full body) using evidence-backed compound lifts + key accessories; fit into 60–90 min 
with sets/reps/rest.
Todo #3: Design Workout B (full body) with different primary patterns/exercises to balance fatigue and coverage; 
fit into 60–90 min.
Todo #4: Provide weight prescriptions using %1RM / RPE plus example loads (based on reasonable estimated 1RMs) and 
clear progression guidance.
Todo #5: Format final plan in Rich console markup with sets, reps, weight, rest, and brief notes for 50+ 
considerations (joint-friendly, volume).

...

Weight guidance created: primary prescription via %1RM/RPE plus example loads using reasonable estimated 
experienced 50+ male baseline 1RMs (for illustration): Squat 300 lb, Bench 225, Deadlift 365, OHP 145, Row 205, 
Pulldown 180, Leg press 500. Example working weights computed from %1RM targets; included instruction to replace 
with personal loads using RPE (keep 1–3 RIR) and double-progression (add reps then small load jumps 2.5–10 lb when 
top of range is met at target RPE).

Todo #1: Set assumptions for an experienced 50+ lifter (weekly frequency, intensity targets, RPE/RIR, warm-up, rest
times) and define how to prescribe weights without exact 1RMs.
Todo #2: Design Workout A (full body) using evidence-backed compound lifts + key accessories; fit into 60–90 min 
with sets/reps/rest.
Todo #3: Design Workout B (full body) with different primary patterns/exercises to balance fatigue and coverage; 
fit into 60–90 min.
Todo #4: Provide weight prescriptions using %1RM / RPE plus example loads (based on reasonable estimated 1RMs) and 
clear progression guidance.
Todo #5: Format final plan in Rich console markup with sets, reps, weight, rest, and brief notes for 50+ 
considerations (joint-friendly, volume).

Final response formatted in Rich console markup: includes schedule, warm-up, workouts A/B with exercises, 
sets/reps/rest, weight as %1RM + example lb, and brief 50+ notes (joint-friendly exercise swaps, tempo/control, 
deload guidance).

Todo #1: Set assumptions for an experienced 50+ lifter (weekly frequency, intensity targets, RPE/RIR, warm-up, rest
times) and define how to prescribe weights without exact 1RMs.
Todo #2: Design Workout A (full body) using evidence-backed compound lifts + key accessories; fit into 60–90 min 
with sets/reps/rest.
Todo #3: Design Workout B (full body) with different primary patterns/exercises to balance fatigue and coverage; 
fit into 60–90 min.
Todo #4: Provide weight prescriptions using %1RM / RPE plus example loads (based on reasonable estimated 1RMs) and 
clear progression guidance.
Todo #5: Format final plan in Rich console markup with sets, reps, weight, rest, and brief notes for 50+ 
considerations (joint-friendly, volume).

2-Day Full-Body Plan (50+, experienced) — Muscle Maintenance (primary) + Hypertrophy (secondary)

Schedule: 2 non-consecutive days (e.g., Mon = Workout A, Thu = Workout B)  
Session length: ~60–90 min including warm-up  
Effort target: most work sets at RPE 7–9 (leave ~1–3 reps in reserve)  
Rest: big compounds 2–3 min; accessories 60–90 sec  
Warm-up (8–12 min): easy cardio + hips/shoulders mobility, then 2–4 ramp sets for the first 1–2 lifts.

How to choose the weight
- Use either:
  - %1RM guide (below), or
  - pick a load that lands you in the rep range at RPE 7–9.
- Progression (double progression): when you hit the top of the rep range for all sets at the target RPE, add load 
next time (upper body +2.5–5 lb; lower body +5–10 lb; machines +1 plate/small increment).

Example loads below assume these illustrative 1RMs for an experienced lifter (adjust to yours):  
Squat 300 lb, Bench 225, Deadlift 365, OHP 145, Row 205, Lat Pulldown 180, Leg Press 500.

---

Workout A (Full Body)
Primary emphasis: squat + horizontal press + vertical pull

1) Squat pattern (choose one)  
- High-bar back squat or Safety-bar squat  
  3–4 sets x 5–6 reps @ ~80–85% 1RM (RPE 7–9) | Rest 2–3 min  
  Example: 240–255 lb (if 1RM squat = 300)  
  Joint-friendlier swap: Hack squat or Leg press if needed.

2) Barbell bench press (or DB bench if shoulders prefer)  
- 3–4 x 6–8 @ ~75–82% 1RM | Rest 2–3 min  
  Example: 170–185 lb (if 1RM bench = 225)

3) Lat pulldown (neutral or medium grip)  
- 3 x 8–12 @ ~65–75% 1RM | Rest 90–120 sec  
  Example: 115–135 lb (if 1RM pulldown = 180)

4) Romanian deadlift (barbell or DB)  
- 3 x 6–10 @ ~70–80% (of your RDL 1RM or equivalent RPE) | Rest 2 min  
  Example proxy: ~225–275 lb for many lifters (choose load for RPE 7–8)

5) Chest-supported row machine (or chest-supported DB row)  
- 2–3 x 8–12 @ RPE 8 | Rest 90 sec  
  Load: choose a weight that leaves ~1–2 reps in reserve on each set.

6) Lateral raise (DB or cable)  
- 2–3 x 12–20 @ RPE 8–9 | Rest 60–75 sec

7) Standing calf raise (machine)  
- 2–3 x 8–12 with a controlled 1–2 sec pause at the stretch | Rest 60–90 sec  
  Load: RPE 8–9.

Optional time-saver: Superset #6 and #7.

---

Workout B (Full Body)
Primary emphasis: hinge + vertical press + horizontal row + unilateral legs

1) Deadlift pattern (choose one)  
- Trap-bar deadlift (often most joint-friendly) or Conventional deadlift  
  3 x 3–5 @ ~82–88% 1RM (RPE 7–9) | Rest 2–3+ min  
  Example: 300–320 lb (if 1RM deadlift = 365)  
  If fatigue/soreness is high: do 4–5 sets of 3 @ RPE ~7 (crisp reps).

2) Overhead press (barbell or DB)  
- 3 x 5–8 @ ~70–80% 1RM | Rest 2 min  
  Example: 100–115 lb (if 1RM OHP = 145)

3) One-arm DB row (or seated cable row if low back prefers)  
- 3 x 8–12 each side @ RPE 8 | Rest 90 sec  
  Load: choose a DB that leaves ~1–2 reps in reserve.

4) Unilateral leg work (choose one)  
- Bulgarian split squat (DB) or Leg press (single-leg or standard)  
  - Split squat: 2–3 x 8–12 each leg @ RPE 8 | Rest 90 sec  
  - Leg press: 3 x 10–15 @ RPE 8 | Rest 2 min  
  Example leg press: 325–375 lb for sets of 10–15 (if 1RM leg press = 500, ~65–75%)

5) Incline DB press (or machine incline press)  
- 2–3 x 8–12 @ RPE 8 | Rest 90 sec  
  Load: pick DBs you can control without shoulder irritation.

6) Hamstring curl machine (seated preferred for many)  
- 2–3 x 10–15 @ RPE 8–9 | Rest 60–90 sec

7) Cable Pallof press (anti-rotation) or loaded carry  
- Pallof: 2–3 x 10–15/side (2-sec hold) | Rest 45–60 sec  
- Farmer carry: 4–6 x 20–40 m with challenging DB/KBs | Rest 60–90 sec

---

50+ execution notes (to keep results high and joints happy)
- Prioritize controlled eccentrics (2–3 sec down) on most lifts; avoid grinders weekly.
- If a joint complains: swap to safety-bar squat, trap-bar deadlift, DB presses, chest-supported rows, machines 
(stimulus stays high with less irritation).
- Every 4–8 weeks, take a deload week: cut sets ~30–40% and keep RPE ~6–7.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>